# 1. Configuración del Entorno y Carga de Datos
En esta sección se importan las librerías necesarias (`pandas`, `os`, `glob`) y se establece la conexión con la carpeta de datos procesados. Se realiza una carga masiva de los históricos de incendios para consolidar el dataset inicial.

In [6]:
import pandas as pd
import glob
import os

# Configurar ruta a tus procesados
ruta_procesados = os.path.join(os.path.dirname(os.getcwd()), "02_Procesados")
archivos_csv = glob.glob(os.path.join(ruta_procesados, "*.csv"))

# Leer y concatenar todos los CSVs
df = pd.concat((pd.read_csv(f, low_memory=False) for f in archivos_csv), ignore_index=True)

print(f"Dataset cargado con {df.shape[0]} filas y {df.shape[1]} columnas.")

Dataset cargado con 221839 filas y 155 columnas.


# 2. Diagnóstico de Calidad y Auditoría de Nulos
Antes de cualquier transformación, realizamos un análisis de integridad. Esta celda genera un reporte detallado del porcentaje de valores nulos por columna. Esto nos permite tomar decisiones técnicas basadas en datos sobre qué variables descartar o imputar.

In [7]:
import pandas as pd
import os

# 1. Cargamos los datos (Asegúrate de que la ruta sea correcta según tu estructura)
ruta_script = os.getcwd()
ruta_base = os.path.dirname(ruta_script)
ruta_archivo = os.path.join(ruta_base, "02_Procesados", "hechos_incendios_2023.csv")

if os.path.exists(ruta_archivo):
    df = pd.read_csv(ruta_archivo, encoding='utf-8-sig')
    
    # 2. Creamos un reporte detallado de las 155 columnas
    reporte = pd.DataFrame({
        'Columna': df.columns,
        'Tipo_Dato': df.dtypes.values,
        'Valores_No_Nulos': df.count().values,
        'Nulos': df.isnull().sum().values,
        '%_Nulos': (df.isnull().sum().values / len(df) * 100).round(2)
    })

    # 3. Ordenamos por porcentaje de nulos para ver qué columnas están más vacías
    reporte_ordenado = reporte.sort_values(by='%_Nulos', ascending=False)

    # 4. Mostramos el reporte (configuramos pandas para que no trunque las filas)
    with pd.option_context('display.max_rows', None, 'display.max_columns', None):
        print(f"Resumen de calidad de datos (Total filas: {len(df)}):")
        print(reporte_ordenado)
else:
    print("❌ No se encontró el archivo CSV en la ruta especificada. Revisa la carpeta 02_Procesados.")

Resumen de calidad de datos (Total filas: 201):
                               Columna Tipo_Dato  Valores_No_Nulos  Nulos  \
117                  detectadoporotros    object                 2    199   
111                         llegadapac    object                 3    198   
115               probabilidadignicion   float64                 4    197   
116                          idpeligro   float64                 4    197   
110              idtipoataqueindirecto   float64                 7    194   
112                         causaotros    object                 9    192   
109            idincidenciaproteccivil   float64                 9    192   
114           afectadourbanoforestalsi   float64                 9    192   
107                       idnoforestal   float64                14    187   
89                     motivacionotros    object                30    171   
98                       brigadastrans   float64                43    158   
99                          

# 3. Refinamiento Masivo y Limpieza de Datos
Aquí aplicamos las reglas de negocio para "sanear" el dataset:
* **Imputación**: Rellenamos con ceros las métricas numéricas vacías.
* **Estandarización**: Eliminamos espacios en blanco innecesarios en los campos de texto.
* **Filtrado**: Eliminamos columnas que superan el umbral crítico de nulos (90%).

In [8]:
import pandas as pd
import os
import glob

def limpiar_dataframe(df, umbral_nulos=0.90):
    """
    Se encarga exclusivamente de la lógica de transformación de datos.
    """
    total_filas = len(df)
    
    # 1. Filtrar columnas por nulos
    pct_nulos = df.isnull().sum() / total_filas
    columnas_a_mantener = pct_nulos[pct_nulos <= umbral_nulos].index.tolist()
    df = df[columnas_a_mantener].copy()
    
    # 2. Imputar ceros en métricas numéricas
    cols_numericas = df.select_dtypes(include=['float64', 'int64']).columns
    df[cols_numericas] = df[cols_numericas].fillna(0)
    
    # 3. Limpieza de textos (Trim)
    cols_texto = df.select_dtypes(include=['object']).columns
    for col in cols_texto:
        df[col] = df[col].astype(str).str.strip()
        
    return df

def ejecutar_refinamiento_masivo():
    ruta_script = os.getcwd()
    ruta_base = os.path.dirname(ruta_script)
    ruta_input = os.path.join(ruta_base, "02_Procesados")
    ruta_output = os.path.join(ruta_base, "03_Refinados")

    if not os.path.exists(ruta_output):
        os.makedirs(ruta_output)

    archivos_csv = glob.glob(os.path.join(ruta_input, "hechos_incendios_*.csv"))

    for archivo in archivos_csv:
        nombre = os.path.basename(archivo)
        print(f"Procesando: {nombre}")
        
        # Añadimos low_memory=False para evitar el DtypeWarning
        df = pd.read_csv(archivo, encoding='utf-8-sig', low_memory=False)
        
        # Transformación
        df_limpio = limpiar_dataframe(df)
        
        # Guardado
        ruta_final = os.path.join(ruta_output, f"limpio_{nombre}")
        df_limpio.to_csv(ruta_final, index=False, encoding='utf-8-sig')
        print(f"   ✅ Columnas finales: {len(df_limpio.columns)}")

ejecutar_refinamiento_masivo()

Procesando: hechos_incendios_2005.csv
   ✅ Columnas finales: 105
Procesando: hechos_incendios_2006.csv
   ✅ Columnas finales: 103
Procesando: hechos_incendios_2007.csv
   ✅ Columnas finales: 99
Procesando: hechos_incendios_2008.csv
   ✅ Columnas finales: 99
Procesando: hechos_incendios_2009.csv
   ✅ Columnas finales: 102
Procesando: hechos_incendios_2010.csv
   ✅ Columnas finales: 105
Procesando: hechos_incendios_2011.csv
   ✅ Columnas finales: 106
Procesando: hechos_incendios_2012.csv
   ✅ Columnas finales: 107
Procesando: hechos_incendios_2013.csv
   ✅ Columnas finales: 106
Procesando: hechos_incendios_2014.csv
   ✅ Columnas finales: 104
Procesando: hechos_incendios_2015.csv
   ✅ Columnas finales: 108
Procesando: hechos_incendios_2016.csv
   ✅ Columnas finales: 128
Procesando: hechos_incendios_2017.csv
   ✅ Columnas finales: 129
Procesando: hechos_incendios_2018.csv
   ✅ Columnas finales: 128
Procesando: hechos_incendios_2019.csv
   ✅ Columnas finales: 128
Procesando: hechos_incendio

# 4. Identificación de Columnas Comunes (Intersección)
Para garantizar la estabilidad del modelo histórico (2018-2023), identificamos las columnas que están presentes en todos los años. Esto asegura que el esquema de datos sea consistente y que Power BI no encuentre errores de estructura al actualizar los datos.

In [9]:
import glob
import pandas as pd
import os

def encontrar_columnas_comunes():
    ruta_base = os.path.dirname(os.getcwd())
    ruta_input = os.path.join(ruta_base, "03_Refinados")
    archivos = glob.glob(os.path.join(ruta_input, "limpio_hechos_incendios_*.csv"))
    
    listas_columnas = []
    
    for archivo in archivos:
        # Solo leemos la cabecera para ir rápido
        df_temp = pd.read_csv(archivo, nrows=0, encoding='utf-8-sig')
        listas_columnas.append(set(df_temp.columns))
    
    # Encontramos la intersección (columnas presentes en TODOS los sets)
    columnas_comunes = list(set.intersection(*listas_columnas))
    columnas_comunes.sort()
    
    print(f"✅ Columnas comunes encontradas: {len(columnas_comunes)}")
    return columnas_comunes

# Ejecutamos para ver el resultado
columnas_finales = encontrar_columnas_comunes()
print(columnas_finales)

✅ Columnas comunes encontradas: 85
['afectoespacionnatprot', 'afectotierraagrariarefores', 'anio', 'brigadastrans', 'conaprovechamientototal', 'controlado', 'cuadricula', 'daniosconaprovechamiento', 'daniossinaprovechamiento', 'descargas', 'deteccion', 'diasultimalluvia', 'direccionviento', 'esconsorciado', 'esprotector', 'estadomasaf', 'estadomasal', 'estadomasamb', 'estadomasar', 'extinguido', 'fcc', 'hoja', 'hora', 'humrelativa', 'huso', 'idalteracionpaisaje', 'idataque', 'idcatalogomonte', 'idcausa', 'idcausante', 'idcertidumbrecausa', 'idclasedia', 'idcomarcaisla', 'idcomunidad', 'iddemanialmonte', 'iddetectadopor', 'idefectoeneconomia', 'idefectovidasilvestre', 'identidadmenor', 'idespecie', 'idestacionmeteorologica', 'idestadocampaniaprovincia', 'idestadopif', 'idgrupomedioretardante', 'idiniciadojuntoa', 'idmedioaereo', 'idmediopersonalext', 'idmediopesado', 'idmodelocombustion', 'idmotivacion', 'idmunicipio', 'idnoarboladoherbaceo', 'idnoarboladolenioso', 'idpartemonte', 'idpi

# 5. Generación de Archivos de Staging (Consolidados)
En esta etapa de "Staging", creamos archivos físicos estandarizados en la carpeta `04_Consolidados`. Cada año se recorta exactamente a las mismas columnas, dejando los datos listos para el modelado final por tablas lógicas.

In [10]:
def generar_archivos_finales(columnas_comunes):
    import os
    import glob
    import pandas as pd

    ruta_base = os.path.dirname(os.getcwd())
    ruta_input = os.path.join(ruta_base, "03_Refinados")
    ruta_output = os.path.join(ruta_base, "04_Consolidados")
    
    if not os.path.exists(ruta_output):
        os.makedirs(ruta_output)
        print(f"Carpeta creada: {ruta_output}")

    archivos = glob.glob(os.path.join(ruta_input, "limpio_hechos_incendios_*.csv"))
    
    for archivo in archivos:
        nombre = os.path.basename(archivo)
        # Leemos con low_memory=False para asegurar consistencia de tipos
        df = pd.read_csv(archivo, encoding='utf-8-sig', low_memory=False)
        
        # Filtramos para quedarnos SOLO con las comunes
        df_final = df[columnas_comunes].copy()
        
        # Guardamos en la carpeta final
        ruta_final = os.path.join(ruta_output, nombre)
        df_final.to_csv(ruta_final, index=False, encoding='utf-8-sig')
        print(f"✅ {nombre} unificado a {len(df_final.columns)} columnas.")

# Usamos la variable 'columnas_finales' que generaste en el paso anterior
generar_archivos_finales(columnas_finales)

✅ limpio_hechos_incendios_2005.csv unificado a 85 columnas.
✅ limpio_hechos_incendios_2006.csv unificado a 85 columnas.
✅ limpio_hechos_incendios_2007.csv unificado a 85 columnas.
✅ limpio_hechos_incendios_2008.csv unificado a 85 columnas.
✅ limpio_hechos_incendios_2009.csv unificado a 85 columnas.
✅ limpio_hechos_incendios_2010.csv unificado a 85 columnas.
✅ limpio_hechos_incendios_2011.csv unificado a 85 columnas.
✅ limpio_hechos_incendios_2012.csv unificado a 85 columnas.
✅ limpio_hechos_incendios_2013.csv unificado a 85 columnas.
✅ limpio_hechos_incendios_2014.csv unificado a 85 columnas.
✅ limpio_hechos_incendios_2015.csv unificado a 85 columnas.
✅ limpio_hechos_incendios_2016.csv unificado a 85 columnas.
✅ limpio_hechos_incendios_2017.csv unificado a 85 columnas.
✅ limpio_hechos_incendios_2018.csv unificado a 85 columnas.
✅ limpio_hechos_incendios_2019.csv unificado a 85 columnas.
✅ limpio_hechos_incendios_2020.csv unificado a 85 columnas.
✅ limpio_hechos_incendios_2021.csv unifi

# 6. Pipeline Final: Modelado Relacional y Generación de Geo_Key
Esta es la fase de arquitectura de datos. El script automatiza tres tareas críticas:
1. **Modelado**: Reparte las 108 columnas en tablas lógicas (Hechos, Logística, Tiempos, etc.) siguiendo un esquema en estrella.
2. **Geo_Key**: Genera la llave única `Provincia-Municipio` para conectar automáticamente con la dimensión geográfica.
3. **Exportación**: Genera los CSV definitivos listos para ser consumidos por Power BI.

In [12]:
import pandas as pd
import os
import glob

# --- CONFIGURACIÓN DE RUTAS ---
ruta_input = r"C:\Users\usuario\Documents\IT Academy\Fuego\Scrapping\04_Consolidados"
ruta_output = r"C:\Users\usuario\Documents\IT Academy\Fuego\Scrapping\05_Final_PowerBI"

# --- EL MAPEO TOTAL (LAS 85 COLUMNAS CLASIFICADAS) ---
MAPEO_MAESTRO = {
    "Fact_Pif": {
        # Identificación y Estados
        "idpif": "IdPif", "numeroparte": "NumeroParte", "anio": "anio", "numero": "numero_registro",
        "idestadopif": "IdEstadoPif", "idestadocampaniaprovincia": "IdEstadoCampania", 
        
        # EL PELIGRO (Rescatado de pif_comun / Portfolio)
        "idpeligro": "Id_Peligro",               # <--- NUEVA: Riesgo Meteorológico (0-4)
        "probabilidadignicion": "Prob_Ignicion", # <--- NUEVA: Probabilidad técnica de ignición
        
        # Calendario y Meteo
        "idclasedia": "Id_Tipo_Dia",             # <--- CORREGIDA: Laborable/Festivo/Sábado
        "tempmaxima": "temp_maxima", "humrelativa": "hum_relativa", 
        "velocidadviento": "velocidad_viento", "direccionviento": "direccion_viento",
        "diasultimalluvia": "dias_ultima_lluvia", "idestacionmeteorologica": "IdEstacionMeteo",

        # Causas
        "idcausa": "IdCausa", "idmotivacion": "IdMotivacion",
        "idcertidumbrecausa": "IdCertidumbreCausa", "idcausante": "IdCausante", "idtipofuego": "IdTipoFuego",
        "idiniciadojuntoa": "IdIniciadoJuntoA", "idpartemonte": "id_parte_monte"
    },
    "Dim_Geografia": {
        # Ubicación y Coordenadas
        "idpif": "IdPif", "idcomunidad": "IDcomunidad", "idprovincia": "IDprovincia", 
        "idmunicipio": "IDmunicipio", "idcomarcaisla": "IDcomarca_isla", "paraje": "paraje",
        "identidadmenor": "identidad_menor", "nummunicipiosafectados": "num_municipios_afectados",
        "x": "Coord_X", "y": "Coord_Y", "huso": "huso", "hoja": "hoja_mapa", "cuadricula": "cuadricula"
    },
    "Fact_Operativa": {
        # Logística, Medios y Tiempos de respuesta
        "idpif": "IdPif", "hora": "hora_inicio", "deteccion": "f_deteccion", "controlado": "f_controlado",
        "extinguido": "f_extinguido", "primeranotificaciondesde112": "aviso_112", "llegadapbh": "llegada_p_helitrans",
        "llegadapmae": "llegada_m_aereos", "llegadapmt": "llegada_m_terrestres", "brigadastrans": "num_brigadas",
        "descargas": "num_descargas", "idmedioaereo": "id_medio_aereo", "idmediopersonalext": "id_medio_personal_ext",
        "idmediopesado": "id_medio_pesado", "idtransportepersonal": "id_transporte_personal",
        "idtitularidadmedio": "id_titularidad_medio", "idvigilantefijo": "id_vigilante_fijo",
        "iddetectadopor": "id_detectado_por", "idataque": "id_ataque", "idgrupomedioretardante": "id_grupo_retardante"
    },
    "Fact_Territorio": {
        # Naturaleza, Vegetación y Superficies
        "idpif": "IdPif", "idespecie": "IdEspecie", "superficie": "Superficie_Total", 
        "superficiearbolada": "Sup_Arbolada_Manual", "superficiearboladatotal": "Sup_Arbolada_Total",
        "superficienoarbolada": "Sup_No_Arbolada_Manual", "superficienoarboladatotal": "Sup_No_Arbolada_Total",
        "conaprovechamientototal": "Sup_Con_Aprovechamiento", "sinaprovechamientototal": "Sup_Sin_Aprovechamiento",
        "afectoespacionnatprot": "Es_Zona_Protegida", "esconsorciado": "es_consorciado", "esprotector": "es_protector",
        "idcatalogomonte": "id_catalogo_monte", "iddemanialmonte": "id_demanial_monte", "idtitularidadmonte": "id_titularidad_monte",
        "idtipoarea": "id_tipo_area", "idmodelocombustion": "id_modelo_combustion", "idriesgoerosion": "id_riesgo_erosion",
        "idalteracionpaisaje": "id_alteracion_paisaje", "idporcentajeautoregenerable": "id_regeneracion_prog",
        "fcc": "fcc_densidad", "idnoarboladoherbaceo": "id_herbaceo", "idnoarboladolenioso": "id_lenioso",
        "estadomasaf": "estado_masa_f", "estadomasal": "estado_masa_l", "estadomasamb": "estado_masa_mb", 
        "estadomasar": "estado_masa_r", "idefectovidasilvestre": "id_efecto_vida_silvestre", "afectotierraagrariarefores": "afecto_agraria_refores"
    },
    "Fact_Economia": {
        # Valoración económica y Daños
        "idpif": "IdPif", "totalperdidas": "Total_Perdidas_Euros", "idefectoeneconomia": "id_efecto_economia",
        "perjuiciosconaprovechamiento": "Perjuicios_Aprovechamiento", "perjuiciossinaprovechamiento": "Perjuicios_Sin_Aprovechamiento",
        "daniosconaprovechamiento": "Danios_Restauracion_Aprov", "daniossinaprovechamiento": "Danios_Restauracion_Sin_Aprov"
    }
}

def pipeline_incendios_final():
    if not os.path.exists(ruta_output): os.makedirs(ruta_output)
    archivos = glob.glob(os.path.join(ruta_input, "*.csv"))
    print(f"🚀 Iniciando ETL Maestro. Procesando {len(archivos)} archivos consolidados...")

    for nombre_tabla, mapeo in MAPEO_MAESTRO.items():
        acumulado = []
        for f in archivos:
            df_raw = pd.read_csv(f, low_memory=False, dtype=str)
            cols_disponibles = [c for c in mapeo.keys() if c in df_raw.columns]
            
            if cols_disponibles:
                df_temp = df_raw[cols_disponibles].copy().rename(columns=mapeo)
                
                # Generación de Geo_Key (Lógica para Dim_Geografia)
                if nombre_tabla == "Dim_Geografia" and 'IDprovincia' in df_temp.columns:
                    df_temp['Geo_Key'] = df_temp['IDprovincia'].str.zfill(2) + "-" + df_temp['IDmunicipio'].str.zfill(3)
                
                acumulado.append(df_temp)
        
        if acumulado:
            df_final = pd.concat(acumulado, ignore_index=True).drop_duplicates()
            df_final.to_csv(os.path.join(ruta_output, f"{nombre_tabla}.csv"), index=False, encoding='utf-8-sig')
            print(f"✅ Tabla {nombre_tabla} generada. Columnas: {len(df_final.columns)} | Filas: {len(df_final)}")

if __name__ == "__main__":
    pipeline_incendios_final()

🚀 Iniciando ETL Maestro. Procesando 19 archivos consolidados...
✅ Tabla Fact_Pif generada. Columnas: 20 | Filas: 221839
✅ Tabla Dim_Geografia generada. Columnas: 14 | Filas: 221839
✅ Tabla Fact_Operativa generada. Columnas: 20 | Filas: 221839
✅ Tabla Fact_Territorio generada. Columnas: 29 | Filas: 221839
✅ Tabla Fact_Economia generada. Columnas: 7 | Filas: 221839


In [5]:
import pandas as pd
import glob

# Ruta a tus consolidados
ruta = r"C:\Users\usuario\Documents\IT Academy\Fuego\Scrapping\04_Consolidados\*.csv"
archivos = glob.glob(ruta)

if archivos:
    # Leemos solo la primera fila del primer archivo
    test_df = pd.read_csv(archivos[0], nrows=0)
    print("🔍 Columnas reales encontradas en el origen:")
    print(test_df.columns.tolist())
    
    if 'idclasedia' in test_df.columns:
        print("\n✅ 'idclasedia' EXISTE. El script debería mapearlo a ID_peligro.")
    else:
        print("\n❌ 'idclasedia' NO EXISTE. Tenemos que buscar otro nombre en la lista de arriba.")

🔍 Columnas reales encontradas en el origen:
['afectoespacionnatprot', 'afectotierraagrariarefores', 'anio', 'brigadastrans', 'conaprovechamientototal', 'controlado', 'cuadricula', 'daniosconaprovechamiento', 'daniossinaprovechamiento', 'descargas', 'deteccion', 'diasultimalluvia', 'direccionviento', 'esconsorciado', 'esprotector', 'estadomasaf', 'estadomasal', 'estadomasamb', 'estadomasar', 'extinguido', 'fcc', 'hoja', 'hora', 'humrelativa', 'huso', 'idalteracionpaisaje', 'idataque', 'idcatalogomonte', 'idcausa', 'idcausante', 'idcertidumbrecausa', 'idclasedia', 'idcomarcaisla', 'idcomunidad', 'iddemanialmonte', 'iddetectadopor', 'idefectoeneconomia', 'idefectovidasilvestre', 'identidadmenor', 'idespecie', 'idestacionmeteorologica', 'idestadocampaniaprovincia', 'idestadopif', 'idgrupomedioretardante', 'idiniciadojuntoa', 'idmedioaereo', 'idmediopersonalext', 'idmediopesado', 'idmodelocombustion', 'idmotivacion', 'idmunicipio', 'idnoarboladoherbaceo', 'idnoarboladolenioso', 'idpartemont

In [8]:
import pandas as pd
import glob
import os

ruta_in = r"C:\Users\usuario\Documents\IT Academy\Fuego\Scrapping\04_Consolidados\*.csv"
ruta_out = r"C:\Users\usuario\Documents\IT Academy\Fuego\Scrapping\05_Final_PowerBI\Rel_Factor_Calculo_Perdida_Parte_Monte.csv"

# Mapeo ampliado con los campos reales que has encontrado
mapeo_economico = {
    "idpif": "IdPif",
    "numeroparte": "NumeroParte",
    "idespecie": "IdEspecie",
    "superficie": "Superficie",
    "totalperdidas": "Total_Perdidas",
    "perjuiciosconaprovechamiento": "Perjuicios_Aprovechamiento",
    "perjuiciossinaprovechamiento": "Perjuicios_Sin_Aprovechamiento",
    "daniosconaprovechamiento": "Danios_Aprovechamiento",
    "daniossinaprovechamiento": "Danios_Sin_Aprovechamiento"
}

archivos = glob.glob(ruta_in)
acumulado = []

print("💰 Extrayendo datos económicos y de superficies...")

for f in archivos:
    df = pd.read_csv(f, low_memory=False, dtype=str)
    # Extraemos solo si existen en el CSV
    columnas_reales = [c for c in mapeo_economico.keys() if c in df.columns]
    
    if columnas_reales:
        df_temp = df[columnas_reales].rename(columns=mapeo_economico)
        acumulado.append(df_temp)

if acumulado:
    df_final = pd.concat(acumulado, ignore_index=True).drop_duplicates()
    df_final.to_csv(ruta_out, index=False, encoding='utf-8-sig')
    print(f"✅ Tabla económica mejorada con {len(df_final.columns)} columnas y {len(df_final)} filas.")

💰 Extrayendo datos económicos y de superficies...
✅ Tabla económica mejorada con 9 columnas y 221839 filas.
